In [1]:
import collections
import logging
import os
import pathlib
import re
import string
import sys
import time

import numpy as np
import matplotlib.pyplot as plt

import tensorflow_datasets as tfds
import tensorflow_text as text
import tensorflow as tf

In [2]:
logging.getLogger('tensorflow').setLevel(logging.ERROR)  # suppress warnings

In [3]:
from pathlib import Path

def load_data(path):
    path = Path(path)  # Convert string to Path object
    text = path.read_text(encoding='utf-8')

    lines = text.splitlines()
    pairs = [line.split('\t') for line in lines]

    inp = [inp for targ, inp in pairs]
    targ = [targ for targ, inp in pairs]

    return inp, targ

def load_data_as_dataset(path):
    targ, inp= load_data(path)
    dataset = tf.data.Dataset.from_tensor_slices((inp, targ))
    return dataset

In [4]:
train_examples = load_data_as_dataset('translation_train.txt')


In [5]:
val_examples = load_data_as_dataset('translation_val.txt')


In [6]:
for  dyu_examples, fr_examples in train_examples.batch(3).take(1):
    for fr in fr_examples.numpy():
        print(fr.decode('utf-8'))

    print()

    for dyu in dyu_examples.numpy():
        print(dyu.decode('utf-8'))

Il boit de l’eau.
Il se plaint toujours.
Quoi ? Quelque chose.

A bi ji min na
A le dalakolontɛ lon bɛ.
Mun? Fɛn dɔ.


2024-07-02 17:19:38.386852: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [ ]:
model_name = "ted_hrlr_translate_dyu_fr_converter"
tokenizers = tf.saved_model.load(model_name)


In [ ]:
[item for item in dir(tokenizers.fr) if not item.startswith('_')]

In [24]:
for fr in fr_examples.numpy():
    print(fr.decode('utf-8'))

Il boit de l’eau.
Il se plaint toujours.
Quoi ? Quelque chose.


In [25]:
encoded = tokenizers.fr.tokenize(fr_examples)

for row in encoded.to_list():
    print(row)

[2, 59, 19, 486, 56, 29, 52, 252, 12, 3]
[2, 59, 88, 33, 299, 298, 302, 12, 3]
[2, 282, 17, 433, 223, 12, 3]


In [26]:
round_trip = tokenizers.fr.detokenize(encoded)
for line in round_trip.numpy():
    print(line.decode('utf-8'))

il boit de l ’ eau .
il se plaint toujours .
quoi ? quelque chose .


In [27]:
tokens = tokenizers.fr.lookup(encoded)
tokens

<tf.RaggedTensor [[b'[START]', b'il', b'b', b'##oit', b'de', b'l', b'\xe2\x80\x99', b'eau',
  b'.', b'[END]']                                                         ,
 [b'[START]', b'il', b'se', b'p', b'##la', b'##int', b'toujours', b'.',
  b'[END]']                                                            ,
 [b'[START]', b'quoi', b'?', b'quelque', b'chose', b'.', b'[END]']]>

In [28]:
def tokenize_pairs(dyu, fr):
    dyu = tokenizers.dyu.tokenize(dyu)
    # Convert from ragged to dense, padding with zeros.
    dyu = dyu.to_tensor()

    fr = tokenizers.fr.tokenize(fr)
    # Convert from ragged to dense, padding with zeros.
    fr = fr.to_tensor()
    return dyu, fr

In [29]:
BUFFER_SIZE = 20000
BATCH_SIZE = 64

In [ ]:
def make_batches(ds):
    return (
            ds
            .cache()
            .shuffle(BUFFER_SIZE)
            .batch(BATCH_SIZE)
            .map(tokenize_pairs, num_parallel_calls=tf.data.AUTOTUNE)
            .prefetch(tf.data.AUTOTUNE))


train_batches = make_batches(train_examples)
val_batches = make_batches(val_examples)